# Aula 00 · Modelos, numpy e matplotlib

Esta aula apresenta o [capítulo 0 do site](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/). A ideia central: **um
modelo diz como uma coisa muda; com isso, dá para prever o futuro um passo de cada
vez** — *próximo = agora + taxa × passo* —, sem fórmula nenhuma. É o método de
Euler, e ele antecipa o curso inteiro: aproximar, medir o erro, diminuir o passo.

**Ao fim da aula você consegue:**

1. explicar o que é um modelo matemático e o que ele deixa de fora;
2. deduzir o passo de Euler a partir da definição de derivada e usá-lo num laço;
3. mostrar, com números, que um passo menor dá um erro menor — e custa mais contas;
4. usar `numpy` para contas com tabelas de números e `matplotlib` para comparar um
   modelo com medições.

**Roteiro:** 🧩 · 1. modelo · 2. 🧑‍🏫 Euler · 3. o passo · 4. numpy · 5. gráficos ·
6. outra área · 🎯 prática · 🧩 a vacina · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Posto de saúde — cadeia de frio das vacinas.**
>
> *Às 18 h, a geladeira de vacinas de um posto de saúde quebrou. As vacinas foram
> para uma caixa térmica, a 4 °C, numa sala a 30 °C. Acima de **8 °C** elas perdem
> a eficácia e têm de ser descartadas. A geladeira nova chega amanhã às 8 h —
> **14 horas depois**. A enfermeira pergunta: "**dá tempo, ou preciso levar as
> vacinas para outro posto agora, à noite?**"*

A caixa térmica esquenta como um café esfria, só que ao contrário. Ninguém vai
resolver a equação no papel às 18 h: no fim da aula, você responde com um laço.

## 1. O que é um modelo matemático

Um paraquedista cai cada vez mais depressa, até o arrasto do ar igualar o peso. O
modelo diz que a aceleração é $\dfrac{dv}{dt} = g - \dfrac{c}{m}v^2$, e o
cálculo acha a fórmula exata:
$v(t) = \sqrt{gm/c}\,\tanh\!\big(\sqrt{gc/m}\;t\big)$.

📖 [capítulo 0 · O que é um modelo matemático](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#o-que-e-um-modelo-matematico)

**✍️ Passo 1.** Defina `g = 9.81`, `m = 68.1`, `c = 0.25` e a função `v(t)` com a fórmula exata (use `np.sqrt` e `np.tanh`). Imprima `v(10)`.

In [ ]:
# ✍️ passo 1

**Preveja:** em 10 segundos de queda, a velocidade passa de 100 km/h (27,8 m/s)?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`49.42` m/s, ou **178 km/h**. Passou, e bem: a gravidade acelera quase 10 m/s
a cada segundo no começo.

</details>

**✍️ Passo 2.** Imprima `v(60)` e `np.sqrt(g * m / c)`.

In [ ]:
# ✍️ passo 2

**Preveja:** depois de um minuto caindo, a velocidade continua crescendo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Os dois dão **51,69 m/s**: é a velocidade **terminal**. O arrasto cresce com
$v^2$ e, quando iguala o peso, a velocidade para de aumentar.

📖 [capítulo 0 · O que é um modelo matemático](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#o-que-e-um-modelo-matematico)

</details>

## 2. Resolver sem fórmula: um passo de cada vez

A fórmula exata só existe porque o modelo é simples. O que existe **sempre** é a
taxa: $g - \frac{c}{m}v^2$ diz quanto a velocidade muda por segundo, agora.

📖 [capítulo 0 · Resolver sem fórmula: um passo de cada vez](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#resolver-sem-formula-um-passo-de-cada-vez)

### 🧑‍🏫 No quadro — o passo de Euler

Caderno de papel aberto. No quadro:

1. a derivada como taxa: $\dfrac{dv}{dt} \approx \dfrac{v_{\text{próximo}} - v_{\text{agora}}}{\Delta t}$;
2. isolar o valor desconhecido: $v_{\text{próximo}} = v_{\text{agora}} + \dfrac{dv}{dt}\,\Delta t$;
3. a taxa vem do modelo e só usa o que já se sabe: $g - \frac{c}{m}v_{\text{agora}}^2$;
4. do valor inicial, repetir: cada passo parte do resultado do anterior;
5. os três primeiros passos à mão, com $\Delta t = 2$ s.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

**Próximo = agora + taxa × passo.** A aproximação do item 1 é a diferença
**progressiva** (capítulo 2), lida ao contrário: lá se conhecem os valores e se
quer a taxa; aqui se conhece a taxa e se querem os valores.

| $t$ | $v$ | taxa | próximo $v$ |
|---|---|---|---|
| 0 | 0,00 | 9,81 | 19,62 |
| 2 | 19,62 | 8,40 | 36,41 |
| 4 | 36,41 | 4,94 | 46,30 |

</details>

**✍️ Passo 3.** Com `v = 0.0` e `dt = 2`, calcule `taxa = g - (c / m) * v**2`, faça `v = v + taxa * dt` e imprima `v`.

In [ ]:
# ✍️ passo 3

**Preveja:** o valor bate com a primeira linha do quadro?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`19.62`. No primeiro passo não há arrasto (o paraquedista está parado), e a
taxa é a gravidade pura: $9{,}81 \times 2$.

</details>

**✍️ Passo 4.** Agora num laço: recomece com `t = 0.0` e `v = 0.0` e, seis vezes, calcule a taxa, avance `v` e `t`, e imprima `t`, `v` e `v(t)` (a exata do passo 1).

In [ ]:
# ✍️ passo 4

**Preveja:** Euler vai errar para cima ou para baixo da exata?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

**Para cima** em todas as linhas (46,30 contra 42,08 em $t = 6$). Cada passo
usa a taxa do **começo** do intervalo, quando o arrasto ainda era menor; o
método acelera demais. Mas as duas colunas chegam à mesma velocidade
terminal.

</details>

> ⚠️ **Armadilha.** Esquecer o `v = ` (escrever só `v + taxa * dt`) não dá erro nenhum: a conta
é feita e jogada fora, e `v` fica em zero para sempre.

### 🎯 Sua vez — Quando passa de 100 km/h?

Escreva `instante_100kmh(dt)`, que parte de `v = 0` e avança o passo de
Euler do paraquedista (`g = 9.81`, `m = 68.1`, `c = 0.25`) **enquanto** a
velocidade for menor que 100 km/h, e devolve o instante `t` em que passou.

In [ ]:
def instante_100kmh(dt):
    # sua solução aqui
    pass

In [ ]:
confere(instante_100kmh, [
    ((1,), 4.0),
    ((0.1,), 3.2000000000000015),
])

<details>
<summary><b>💡 Dica</b></summary>

`while v * 3.6 < 100:` — dentro, o passo de Euler e `t = t + dt`. Crie `v` e
`t` valendo zero **antes** do `while`.

</details>

## 3. Quanto o passo importa

O erro vem de fingir que a taxa fica parada durante o passo inteiro. Com passos
menores, ela tem menos tempo para mudar.

📖 [capítulo 0 · Quanto o passo importa](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#quanto-o-passo-importa)

**✍️ Passo 5.** Calcule a velocidade em `t = 12` com `dt = 1` (12 passos) e com `dt = 0.5` (24 passos), e imprima o erro de cada uma contra `v(12)`.

In [ ]:
# ✍️ passo 5

**Preveja:** com o passo pela metade, o erro cai quanto?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

De cerca de 0,58 para 0,31 m/s: **mais ou menos pela metade**, com o dobro de
contas. Precisão custa trabalho, e cada método tem a sua "taxa de câmbio"
entre os dois — no capítulo 2, ela ganha o nome de **ordem**.

📖 [capítulo 0 · Quanto o passo importa](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#quanto-o-passo-importa)

</details>

## 4. numpy em poucos minutos

Um **array** do `numpy` guarda uma tabela de números, e toda conta com ele vale
para **cada elemento**. As funções `np.exp`, `np.sqrt`, `np.tanh`... também.

📖 [capítulo 0 · numpy em poucos minutos](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#numpy-em-poucos-minutos)

**✍️ Passo 6.** Crie `t = np.linspace(0, 12, 7)` e imprima `t`, `2 * t` e `v(t)`.

In [ ]:
# ✍️ passo 6

**Preveja:** a função `v`, escrita pensando num número só, funciona com o array inteiro?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Funciona: sai a tabela inteira de velocidades de uma vez, porque `np.sqrt` e
`np.tanh` trabalham elemento a elemento. É a mesma tabela do passo 4, sem
laço.

</details>

**✍️ Passo 7.** Imprima `[1, 2, 3] * 2` e depois `np.array([1, 2, 3]) * 2`.

In [ ]:
# ✍️ passo 7

**Preveja:** as duas linhas imprimem a mesma coisa?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não! A lista do Python **repete**: `[1, 2, 3, 1, 2, 3]`. O array
**multiplica**: `[2 4 6]`. Para fazer conta com uma tabela de números, use
array.

</details>

> ⚠️ **Armadilha.** Dados que chegam como lista (lidos de um arquivo, digitados à mão) têm de
virar array **antes** da conta. O erro não avisa: a lista repetida tem o
dobro do tamanho e segue adiante.

## 5. Gráficos com matplotlib

Um gráfico mostra a **forma** que a tabela esconde. Receita: `plt.figure`,
`plt.plot` (linhas, para modelos), `plt.scatter` ou `"o"` (pontos, para medições
ou valores calculados), nomes nos eixos com unidade, legenda e `plt.show()`.

📖 [capítulo 0 · Gráficos com matplotlib](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#graficos-com-matplotlib)

**✍️ Passo 8.** Refaça o laço do passo 4 guardando `t` e `v` em duas listas (`tempos` e `velocidades`, com `append`). Desenhe a exata com `ts = np.linspace(0, 12, 100)` e `plt.plot(ts, v(ts))`, e os pontos de Euler com `plt.plot(tempos, velocidades, "o")`. Ponha nome nos eixos e `plt.show()`.

In [ ]:
# ✍️ passo 8

**Preveja:** os pontos de Euler ficam acima ou abaixo da curva? Onde a diferença é maior?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Acima, com a maior diferença no meio da queda (entre 4 e 8 s), onde a taxa
muda mais depressa. No fim, os dois se juntam na velocidade terminal. O
gráfico mostra de relance o que a tabela só mostrava fazendo contas.

</details>

## 6. Mesmo método, outra área

Um capacitor carregando por um resistor muda a uma taxa
$\dfrac{dV}{dt} = \dfrac{V_f - V}{RC}$. O laço é **o mesmo** do paraquedista;
só muda a linha da taxa.

📖 [capítulo 0 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#mesmo-metodo-outra-area)

**✍️ Passo 9.** Com `Vf = 5.0`, `RC = 1.0`, `V = 0.0` e `dt = 0.1`, faça 10 passos de Euler com a taxa `(Vf - V) / RC` e imprima `V`. Compare com a exata, `Vf * (1 - np.exp(-1))`.

In [ ]:
# ✍️ passo 9

**Preveja:** depois de uma constante de tempo ($t = RC$), o capacitor está com quantos % da tensão final?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Euler dá 3,26 V e a exata, 3,16 V: **63 %** de 5 V. O mesmo laço que derrubou
o paraquedista carrega o capacitor, esfria o café e — já já — esquenta a
caixa de vacinas.

</details>

### 🎯 Sua vez — A tensão chega a 90 %?

Escreva `tempo_ate(fracao, dt)`, que carrega o capacitor (`Vf = 5.0`,
`RC = 1.0`, partindo de `V = 0`) pelo passo de Euler **enquanto** `V` for
menor que `fracao * Vf`, e devolve o instante em que chegou.

A exata é $t = -RC\,\ln(1 - \text{fração})$: para 90 %, $2{,}303$ s.

In [ ]:
def tempo_ate(fracao, dt):
    # sua solução aqui
    pass

In [ ]:
confere(tempo_ate, [
    ((0.9, 0.01), 2.299999999999995),
    ((0.5, 0.01), 0.6900000000000004),
])

<details>
<summary><b>💡 Dica</b></summary>

O mesmo `while` do 🎯 anterior, com a condição `V < fracao * Vf`.

</details>

## 🎯 Prática

Retoma o bloco *2. Resolver sem fórmula*.
📖 [capítulo 0 · Resolver sem fórmula: um passo de cada vez](https://lacouth.github.io/metodos_telecom-site/unidade0-introducao/00-modelos-e-ferramentas/#resolver-sem-formula-um-passo-de-cada-vez)

**O paraquedas abre.** Aos 50 m/s, o paraquedista abre o paraquedas e o
coeficiente de arrasto passa de $0{,}25$ para $12{,}5$ kg/m. Agora a taxa começa
**negativa**: o arrasto é maior que o peso.

### 🎯 Sua vez — O paraquedas aberto

Escreva `velocidade_final(m, c, v0, dt, t_final)`, que começa com
`v = v0` e aplica `round(t_final / dt)` passos de Euler do paraquedista,
devolvendo a velocidade final.

Depois de passar: em quanto tempo ele chega perto da nova velocidade
terminal, $\sqrt{9{,}81 \cdot 68{,}1 / 12{,}5} \approx 7{,}3$ m/s?

In [ ]:
def velocidade_final(m, c, v0, dt, t_final):
    # sua solução aqui
    pass

In [ ]:
confere(velocidade_final, [
    ((68.1, 12.5, 50.0, 0.01, 1), 8.035976744455299),
    ((68.1, 12.5, 50.0, 0.01, 5), 7.310613496064108),
    ((68.1, 0.25, 0.0, 2, 6), 46.29832087760171),
])

<details>
<summary><b>💡 Dica</b></summary>

O laço do passo 4 dentro de uma função, com `v = v0` em vez de `v = 0.0`.

</details>

## 🧩 Resolvendo o problema

> *"**Dá tempo, ou preciso levar as vacinas para outro posto agora, à noite?**"* — a
> enfermeira do posto.

A caixa térmica esquenta pela mesma lei do café: $\dfrac{dT}{dt} = k\,(T_{\text{sala}} - T)$.
A célula 📦 tem os dados.

In [ ]:
# 📦 dados prontos — só rode esta célula
T0 = 4.0          # temperatura inicial das vacinas (°C)
T_SALA = 30.0     # temperatura da sala (°C)
T_LIMITE = 8.0    # acima disso a vacina perde eficácia (°C)
k = 0.012         # constante da caixa térmica (por hora)
PRAZO = 14.0      # horas até a geladeira nova chegar

### 🎯 Sua vez — A vacina na caixa térmica

Escreva `horas_ate_8graus(dt)`, que parte de `T0` e avança o passo de Euler
**enquanto** a temperatura estiver abaixo de `T_LIMITE`, devolvendo o
instante (em horas) em que ela chegou lá. Use as constantes da célula 📦.

In [ ]:
def horas_ate_8graus(dt):
    # sua solução aqui
    pass

In [ ]:
confere(horas_ate_8graus, [
    ((1,), 14.0),
    ((0.5,), 14.0),
    ((0.01,), 13.929999999999747),
])

<details>
<summary><b>💡 Dica</b></summary>

A taxa é `k * (T_SALA - T)`. O `while` é igual ao do 🎯 do capacitor.

</details>

Agora, a resposta para a enfermeira, com três passos diferentes:

In [ ]:
for dt in [1, 0.5, 0.01]:
    print(f"dt = {dt} h: passa de 8 °C depois de", horas_ate_8graus(dt), "h  (prazo:", PRAZO, "h)")

<details>
<summary><b>▶ O que os números dizem</b></summary>

Com `dt = 1` h, Euler diz **14,0 h** — "chega junto com a geladeira, dá justo".
Com `dt = 0.01` h, diz **13,93 h**. A resposta exata do modelo é
$\ln(26/22)/0{,}012 \approx 13{,}92$ h: as vacinas passam de 8 °C **cerca de
cinco minutos antes** de a geladeira chegar.

Numa decisão apertada, o passo grosso deu a resposta **errada**: não porque o erro
fosse grande, mas porque a margem era menor que ele. A resposta para a enfermeira é
levar as vacinas (ou trocar o gelo da caixa) — e a resposta para você é: **antes
de decidir com um número, diminua o passo e veja se ele muda.**

</details>

## 📋 A lista

Abra a [Lista 00](https://lacouth.github.io/metodos_telecom-site/listas/lista00/). O **Exercício 01** é à mão (✏️): os três passos de
Euler do quadro, para você fazer sem olhar. Comece por ele, no papel.

**a)** Qual a taxa no primeiro passo, e por que ela é tão simples?

<details>
<summary><b>▶ Resposta</b></summary>

$9{,}81$ m/s²: com $v = 0$, o arrasto $\frac{c}{m}v^2$ é zero e sobra só a
gravidade.

</details>

**b)** No segundo passo, quanto vale $v^2$? Faça essa conta **antes** de multiplicar por $c/m$.

<details>
<summary><b>▶ Resposta</b></summary>

$19{,}62^2 = 384{,}94$. Separar a conta em pedaços é o que evita o erro mais comum
da prova: esquecer o quadrado.

</details>

Termine o exercício e siga para o **Exercício 02**, a velocidade terminal.

## 🚪 Antes de sair

**1.** O método de Euler do paraquedista errou sempre para cima. No problema da
vacina (a temperatura **subindo** e a taxa **diminuindo**), ele erra para qual lado?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Também para cima: a taxa usada é a do começo do passo, que é a **maior** do
intervalo. Euler esquenta a vacina mais depressa que o modelo — o que, no caso,
deixa a resposta do lado seguro. (Com `dt = 1` o que atrapalhou foi outra coisa: a
resposta só pode ser um número inteiro de horas.)

</details>

**2.** Por que a lista do Python não serve para fazer conta com uma tabela de números?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Porque `*` e `+` com listas **repetem** e **juntam** listas, em vez de fazer a conta
em cada elemento. O array do `numpy` faz a conta elemento a elemento.

</details>

**3.** Um colega diz que o método de Euler não serve para nada, porque sempre erra. O que você responde?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Que todo método numérico erra — e que o importante é **saber quanto** e **como
diminuir**. Com o passo menor, Euler chega tão perto quanto se queira; e ele
funciona justamente nos modelos que **não** têm fórmula exata, que são a maioria.

</details>

## 🏠 Para casa

- Refaça os três passos de Euler do quadro **sem olhar**.
- Termine a [Lista 00](https://lacouth.github.io/metodos_telecom-site/listas/lista00/).
- Leia o começo do [capítulo 1](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/): se o computador
  faz contas com 16 algarismos, de onde vêm os erros?